<a href="https://colab.research.google.com/github/Abeer-24/NLP/blob/main/Text_Generation_using_GPT_2_on_a_custom_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets accelerate

In [2]:
import torch
from datasets import Dataset
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)

In [3]:
texts = [
    "Artificial intelligence is transforming the world.",
    "Machine learning helps computers learn from data.",
    "Deep learning uses neural networks to solve problems.",
    "Python is widely used for artificial intelligence.",
    "Natural language processing helps computers understand human language.",
    "AI is used in healthcare, education and business.",
    "Data science combines statistics and programming.",
    "Robotics uses artificial intelligence to perform tasks.",
    "Computer vision enables machines to understand images.",
    "The future of technology depends on innovation."
]

dataset = Dataset.from_dict({"text": texts})

print(dataset)

Dataset({
    features: ['text'],
    num_rows: 10
})


In [4]:
tokenizer = GPT2Tokenizer.from_pretrained("openai-community/gpt2")

tokenizer.pad_token = tokenizer.eos_token

print("GPT-2 Tokenizer Loaded Successfully")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

GPT-2 Tokenizer Loaded Successfully


In [5]:
model = GPT2LMHeadModel.from_pretrained(
    "openai-community/gpt2"
)

model.config.pad_token_id = tokenizer.pad_token_id

print("GPT-2 Model Loaded Successfully")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT-2 Model Loaded Successfully


In [6]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=64
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print("Tokenization Completed")

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenization Completed


In [7]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Data Collator Created Successfully")

Data Collator Created Successfully


In [8]:
training_args = TrainingArguments(
    output_dir="./gpt2_custom",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    save_strategy="no",
    logging_steps=1,
    report_to="none",
    fp16=torch.cuda.is_available()
)

In [9]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

print("Trainer Initialized")

Trainer Initialized


In [10]:
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,4.558769
2,4.559693
3,3.153718
4,2.966708
5,2.982558


TrainOutput(global_step=5, training_loss=3.6442890644073485, metrics={'train_runtime': 12.9131, 'train_samples_per_second': 0.774, 'train_steps_per_second': 0.387, 'total_flos': 44909568000.0, 'train_loss': 3.6442890644073485, 'epoch': 1.0})

In [11]:
model.eval()

prompt = "Artificial intelligence"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

device = next(model.parameters()).device
inputs = {key: value.to(device) for key, value in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        pad_token_id=tokenizer.eos_token_id
    )

generated_text = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Generated Text:")
print(generated_text)

Generated Text:
Artificial intelligence is a highly efficient way of solving problems, and its ability to learn is so powerful that it was even invented. The machine was designed to solve problems in a way that was far too complex to comprehend or to solve. But, as computers evolved and


In [12]:
user_prompt = input("Enter a starting sentence: ")

inputs = tokenizer(
    user_prompt,
    return_tensors="pt"
)

inputs = {key: value.to(device) for key, value in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        pad_token_id=tokenizer.eos_token_id
    )

print("\nGenerated Text:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Enter a starting sentence: Artificial intelligence is changing the way humans interact with technology

Generated Text:
Artificial intelligence is changing the way humans interact with technology, and that's probably going to change the way we live. But how can we live without AI?

I think we can. I think we've made lots of good leaps forward in AI, in ways that haven't been possible in the
